In [2]:
from kan import *
import torch.nn as nn
import torch.optim as optim
import torch
from helper_lin import get_all_data
class LassoBrainKAN(nn.Module):
    def __init__(self, hidden_dim=32):
        super(LassoBrainKAN, self).__init__()
        # width=[100, hidden_dim, 1] maps to your 100 inputs, hidden_dim, and 1 output
        # grid and k define the spline granularity and order (3 is standard)
        self.kan = KAN(width=[100, hidden_dim, 1], grid=3, k=3, seed=42) 
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Pass input through KAN, then apply Sigmoid for binary classification
        x = self.kan(x)
        return self.sigmoid(x)

In [3]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
import torch

def evaluate_kan_pipeline(X_avg, y_avg, hidden_dim=32, l1_lambda=0.005, lr=0.01, epochs=100):
    all_test_preds = []
    all_test_labels = []
    all_test_probs = []

    # 5-Fold Stratified CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    print(f"Starting Leakage-Free 5-Fold CV with KAN...")

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_avg, y_avg)):
        # 1. SPLIT DATA FIRST
        X_train_fold, X_test_fold = X_avg[train_idx], X_avg[test_idx]
        y_train_fold, y_test_fold = y_avg[train_idx], y_avg[test_idx]

        # 2. FEATURE SELECTION (Fit ONLY on training data)
        selector = SelectKBest(score_func=f_classif, k=100)
        X_train_reduced = selector.fit_transform(X_train_fold, y_train_fold)
        X_test_reduced = selector.transform(X_test_fold) # Transform only!

        # 3. SCALING (Fit ONLY on training data)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_reduced)
        X_test_scaled = scaler.transform(X_test_reduced) # Transform only!

        # Convert to Tensors
        X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
        y_train_t = torch.tensor(y_train_fold, dtype=torch.float32).view(-1, 1)
        X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)

        # Initialize KAN Model
        model = LassoBrainKAN(hidden_dim=hidden_dim) 
        criterion = torch.nn.BCELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        # Train 
        model.train()
        for epoch in range(epochs):
            optimizer.zero_grad()
            outputs = model(X_train_t)
            bce_loss = criterion(outputs, y_train_t)
            
            # KAN generalized L1 penalty
            l1_penalty = sum(torch.norm(p, 1) for p in model.parameters() if p.dim() > 1)
            total_loss = bce_loss + (l1_lambda * l1_penalty)
            
            total_loss.backward()
            optimizer.step()

        # Evaluate on the unseen Test Fold
        model.eval()
        with torch.no_grad():
            test_probs_t = model(X_test_t)
            preds = (test_probs_t > 0.5).float().numpy().flatten()
            
            all_test_probs.extend(test_probs_t.numpy().flatten())
            all_test_preds.extend(preds)
            all_test_labels.extend(y_test_fold)
        
        print(f"Fold {fold+1} complete.")

    final_acc = accuracy_score(all_test_labels, all_test_preds)
    print(f"\nHonest K-Fold Accuracy: {final_acc * 100:.2f}%")
    
    return all_test_labels, all_test_probs, all_test_preds

X_avg, y_avg = get_all_data()
all_test_labels, all_test_probs, all_test_preds, feature_weights, selected_indices = evaluate_kan_pipeline(X_avg, y_avg)

Starting Leakage-Free 5-Fold CV with KAN...
checkpoint directory created: ./model
saving model version 0.0
Fold 1 complete.
checkpoint directory created: ./model
saving model version 0.0
Fold 2 complete.
checkpoint directory created: ./model
saving model version 0.0
Fold 3 complete.
checkpoint directory created: ./model
saving model version 0.0
Fold 4 complete.
checkpoint directory created: ./model
saving model version 0.0
Fold 5 complete.

Honest K-Fold Accuracy: 56.67%


ValueError: not enough values to unpack (expected 5, got 3)